# 02: Data Understanding - Temporal Trends & Segmentation

This notebook performs supporting exploratory data analysis (EDA). We explore the descriptive features, cancellation audits, temporal purchasing behavior (monthly trends, cohort retention), and simple RFM clustering.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import sys
import os
sys.path.append(os.path.abspath(".."))

from src.db import query_to_dataframe

plt.style.use("seaborn-v0_8-darkgrid")

## 1. Temporal Analysis: Revenue & Orders

In [ ]:
with open("../sql/analysis.sql", "r") as f:
    sql_script = f.read()

queries = sql_script.split(";")
# First query is Monthly Revenue & Order Volume
monthly_trend_query = queries[0].strip()
print(monthly_trend_query)

monthly_df = query_to_dataframe(monthly_trend_query)
display(monthly_df.head())

fig, ax1 = plt.subplots(figsize=(10, 5))
ax2 = ax1.twinx()
ax1.bar(monthly_df["invoice_month"], monthly_df["total_orders"], color="g", alpha=0.6)
ax2.plot(monthly_df["invoice_month"], monthly_df["total_revenue"], color="b", marker="o")
ax1.set_xlabel("Month")
ax1.set_ylabel("Total Orders", color="g")
ax2.set_ylabel("Total Revenue", color="b")
plt.title("Monthly Orders vs Revenue")
plt.xticks(rotation=45)
plt.show()

## 2. Cohort Retention Matrix

In [ ]:
cohort_query = queries[1].strip()
cohort_df = query_to_dataframe(cohort_query)

cohort_pivot = cohort_df.pivot(index="cohort_month", columns="activity_month", values="active_customers")

# Calculate percentages
cohort_sizes = cohort_pivot.iloc[:, 0]
retention_matrix = cohort_pivot.divide(cohort_sizes, axis=0)

plt.figure(figsize=(12, 8))
sns.heatmap(retention_matrix, annot=True, fmt=".0%", cmap="Blues")
plt.title("Cohort Retention Matrix")
plt.show()

## 3. Top Popular Products

In [ ]:
top_products_query = queries[2].strip()
top_products_df = query_to_dataframe(top_products_query)
display(top_products_df.head(10))